# Learning Interpretability Tool 🔥 による言語モデルの解釈可能性

## 概要

このノートブックでは、[Learning Interpretability Tool（🔥LIT）](https://pair-code.github.io/lit/) を使用して、BERT ベースのテキスト分類モデルに対する説明可能性（Explainability）手法を利用する方法を示します。

LIT は、テキスト、画像、表形式データをサポートする、視覚的でインタラクティブな ML モデル理解ツールです。<br>
LIT を使用して、以下のような問いを投げかけ、答えることができます。
- モデルのパフォーマンスが低いのはどのようなサンプルか？
- なぜモデルはこの予測を行ったのか？ 敵対的な挙動や、トレーニング セットからの望ましくない事前分布に起因するものか？
- テキストのスタイル、動詞の時制、代名詞の性別などを変更した場合、モデルは一貫した動作をするか？

LIT には多くの機能が組み込まれていますが、カスタムの解釈可能性手法、評価指標の計算、反実仮想（反事実的）生成、可視化などを追加できるカスタマイズ性も備えています。

このノートブックでは、組み込みの解釈可能性手法である [LIME](#Explore-the-LIME-explanation)、[勾配ノルム（Gradient Norm）](#Gradient-Norm)、[勾配と入力の内積（Gradient-dot-Input）](#Gradient-dot-Input)、および[統合勾配（Integrated Gradients）](#Add-Integrated-Gradients) を探索します。

### 学習目標: 
* Learning Interpretability Tool（🔥LIT）の概要を学ぶ
* LIT の Dataset および Model オブジェクトを構成する
* 解釈可能性手法のために Model をカスタマイズする

## セットアップ

In [ ]:
import os
import warnings

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

In [ ]:
import datetime
import shutil
from collections.abc import Iterable, Sequence

import keras
import keras_hub
import pandas as pd
import tensorflow as tf
from lit_nlp import notebook
from lit_nlp.api import dataset as lit_dataset
from lit_nlp.api import layout
from lit_nlp.api import model as lit_model
from lit_nlp.api import types as lit_types

JsonDict = lit_types.JsonDict
Spec = lit_types.Spec

tf.get_logger().setLevel("ERROR")

In [ ]:
print("Num GPUs Available: ", len(tf.config.list_physical_devices("GPU")))

## BERT ベースの感情分析モデルの構築

このノートブックでは、レビューのテキストに基づいて、映画のレビューを「*positive*（肯定的）」または「*negative*（否定的）」に分類する感情分析モデルをトレーニングします。

[Internet Movie Database](https://www.imdb.com/) からの 50,000 件の映画レビューのテキストを含む [Large Movie Review Dataset](https://ai.stanford.edu/~amaas/data/sentiment/) を使用します。

### IMDB データセットのダウンロード

データセットをダウンロードして展開し、ディレクトリ構造を確認しましょう。

In [ ]:
url = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"

path = "/home/jupyter/"
dataset = keras.utils.get_file(
    "aclImdb_v1.tar.gz", url, untar=True, cache_dir=path, cache_subdir=""
)

dataset_dir = os.path.join(
    os.path.dirname(dataset), "aclImdb_v1_extracted/aclImdb"
)

train_dir = os.path.join(dataset_dir, "train")

# データの読み込みを容易にするために未使用のフォルダを削除します
remove_dir = os.path.join(train_dir, "unsup")
shutil.rmtree(remove_dir)

次に、`text_dataset_from_directory` ユーティリティを使用して、ラベル付きの `tf.data.Dataset` を作成します。

IMDB データセットはすでに train と test に分割されていますが、検証セット（validation set）がありません。下の `validation_split` 引数を使用して、トレーニング データを 80:20 の比率で分割し、検証セットを作成しましょう。

注: `validation_split` 引数と `subset` 引数を使用する場合は、検証用とトレーニング用の分割に重複が生じないように、必ずランダム シードを指定するか、`shuffle=False` を渡してください。

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
batch_size = 32
seed = 42

raw_train_ds = keras.utils.text_dataset_from_directory(
    path + "aclImdb_v1_extracted/aclImdb/train",
    batch_size=batch_size,
    validation_split=0.2,
    subset="training",
    seed=seed,
)

class_names = raw_train_ds.class_names
train_ds = raw_train_ds.cache().prefetch(buffer_size=AUTOTUNE)

val_ds = keras.utils.text_dataset_from_directory(
    path + "aclImdb_v1_extracted/aclImdb/train",
    batch_size=batch_size,
    validation_split=0.2,
    subset="validation",
    seed=seed,
)

val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

test_ds = keras.utils.text_dataset_from_directory(
    path + "aclImdb_v1_extracted/aclImdb/test", batch_size=batch_size
)

test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

### Keras NLP を使用した BERT モデルのトレーニング

このラボの目的のために、感情分析用に事前トレーニング済みの tiny BERT と呼ばれる小さなモデルを読み込みます。tiny BERT は元の BERT と同じ一般的なアーキテクチャを備えていますが、Transformer ブロックの数が少なく、ブロック自体も小さくなっています。

他のモデルタイプについては、[Keras NLP のドキュメント](https://keras.io/api/keras_nlp/models/bert/bert_classifier/) を参照してください。

In [ ]:
classifier = keras_hub.models.BertClassifier.from_preset(
    "bert_tiny_en_uncased_sst2",
)

ウォームアップ期間を設けた AdamW オプティマイザを使用します。BERT のファインチューニングに関する一般的なガイドラインについては、[こちらの論文](https://arxiv.org/abs/1905.05583) を参照してください。

In [ ]:
epochs = 5
steps_per_epoch = tf.data.experimental.cardinality(train_ds).numpy()
num_train_steps = steps_per_epoch * epochs
num_warmup_steps = int(0.1 * num_train_steps)

warmup_tgt = 3e-5
scheduler = keras.optimizers.schedules.CosineDecay(
    0.0,
    num_train_steps,
    warmup_target=warmup_tgt,
    warmup_steps=num_warmup_steps,
)

optimizer = tf.keras.optimizers.AdamW(learning_rate=scheduler)

In [ ]:
classifier.compile(
    optimizer=optimizer,
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[keras.metrics.SparseCategoricalAccuracy()],
)

In [ ]:
FILE_PATH = "./model"

history = classifier.fit(x=train_ds, validation_data=val_ds, epochs=epochs)

---

## Learning Interpretability Tool の使用

Learning Interpretability Tool の使用を開始しましょう。ここでは Keras モデルに LIT を使用しますが、LIT 自体はフレームワークに依存しない（フレームワーク非依存）ことに留意してください。

<img src="https://pair-code.github.io/lit/documentation/_images/lit-system-diagram.svg" width="1084">

このノートブックでは、Dataset と Model という 2 つの LIT オブジェクトを構成します。

### Dataset
Dataset はサンプルのリストであり、LIT の型システムに従った関連する型情報を持っています。

- `spec()` は、各サンプルのフィールドを記述するフラットな辞書（dict）を返す必要があります。各項目は LIT の型を使用して型付けされている点に注意してください。詳細については、[LIT の型に関するドキュメント](https://pair-code.github.io/lit/documentation/api.html#available-types) を参照してください。
- `self._examples` は、`spec()` に準拠したフラットな辞書のリストである必要があります。

ここでは、`tf.data.Dataset` オブジェクトをイテレートすることで、LIT の Dataset に変換します。

LIT は、LIT サーバーに含めたデータセットに読み込まれているすべてのサンプルに対して動作します。そのため、バックエンド サーバーのメモリに収まり、ブラウザに表示できるデータセット サイズを使用するように注意する必要があります。

実装は [Dataset](https://github.com/PAIR-code/lit/blob/main/lit_nlp/api/dataset.py) をサブクラス化する必要があります。通常、以下の例のように数行のコードで記述できます。

In [ ]:
NUM_DATA = 100  # 小さなサンプルを取得
LABELS = ["negative", "positive"]


class ImdbSentimentData(lit_dataset.Dataset):

    def __init__(self, dataset: tf.data.Dataset):
        data_array = list(
            dataset.shuffle(dataset.cardinality()).as_numpy_iterator()
        )
        texts, targets = [], []
        for features, labels in data_array:
            texts.extend([f.decode() for f in features if len(f.decode())])
            targets.extend(labels)
        # self.spec() に準拠した辞書のリストとして保存
        self._examples = [
            {
                "text": text,
                "label": LABELS[1] if target else LABELS[0],
            }
            for text, target in zip(texts[:NUM_DATA], targets[:NUM_DATA])
        ]

    def spec(self):
        return {
            "text": lit_types.TextSegment(),
            "label": lit_types.CategoryLabel(vocab=LABELS),
        }

### Model

Model は、LIT の型システムに従った関連する型情報を持ち、入力を受け取って出力を生成する関数です。

コア API は次の 3 つのメソッドで構成されます。
- `input_spec()`: 必要な入力フィールドを記述したフラットな辞書を返す必要があります。ここでは `"text"` と `"label"` を定義します。
- `output_spec()`: モデルの予測および追加の出力を記述したフラットな辞書を返す必要があります。ここでは `MulticlassPreds` 型を使用します。
- `predict()`: （`input_spec()` を満たす）入力のシーケンスを受け取り、`output_spec()` に一致する並列の出力シーケンスを生成する必要があります。

実装は [Model](https://github.com/PAIR-code/lit/blob/main/lit_nlp/api/model.py) をサブクラス化する必要があります。

このシンプルな Model セットアップにより、入力と出力にのみ着目するブラックボックス解釈可能性手法である LIME を使用できます。

In [ ]:
class BertSentimentModel(lit_model.Model):

    def __init__(self, model: keras.Model):
        self._model = model

    def predict(self, inputs: Iterable[JsonDict]) -> Iterable[JsonDict]:
        examples = [d["text"] for d in inputs]
        tokens = self._model.preprocessor(examples)
        logits = self._model(tokens)
        probs = keras.activations.sigmoid(logits)
        return [{"prob": prob.numpy()} for prob in probs]

    def input_spec(self) -> Spec:
        return {
            "text": lit_types.TextSegment(),
            "label": lit_types.CategoryLabel(required=False, vocab=LABELS),
        }

    def output_spec(self) -> Spec:
        return {
            "prob": lit_types.MulticlassPreds(vocab=LABELS, parent="label"),
        }

それでは、ノートブック内に LIT ウィジェットを作成しましょう。

In [ ]:
# 分析対象のモデルとデータセットを指定して LIT ウィジェットを作成します。
datasets = {"acllmdb": ImdbSentimentData(test_ds)}
models = {"BERT classifier": BertSentimentModel(classifier)}
layouts = {"default": "default"}

# ウィジェットを再実行する際は下のコメントを解除してください。
# widget.stop()
widget = notebook.LitWidget(models, datasets, port=8890)

`render()` メソッドを使用してウィジェットをレンダリングできます。<br>
ウィジェットはノートブック内に埋め込むこともできますが、`open_in_new_tab=True` を追加して別のタブでウィジェットを確認してみましょう。

UI を自由に操作して、上記で定義した概念が LIT UI にどのようにリンクされているか確認してみてください。

In [ ]:
# ウィジェットをレンダリング
widget.render(height=600, open_in_new_tab=True)

### レイアウトのカスタマイズ

UI は使い勝手が良さそうですね。<br>
しかし、レイアウトを少し変更したい場合もあるでしょう。心配いりません。LIT UI はカスタマイズ可能です。<br>

事前構築されたレイアウト カタログから、よりリッチなレイアウトに切り替えてみましょう。<br>
元の UI に戻せるように、辞書で複数のレイアウトを渡して保持します。

In [ ]:
layouts = {
    "rich_layout": layout.THREE_PANEL_LAYOUT,
    "default": notebook.LIT_NOTEBOOK_LAYOUT,
}

In [ ]:
widget.stop()
widget = notebook.LitWidget(models, datasets, layouts=layouts, port=8890)
widget.render(height=600, open_in_new_tab=True)

UI をカスタマイズしたい場合は、[ドキュメント](https://pair-code.github.io/lit/documentation/api.html#customizing-the-layout) および先ほど使用した [UI 定義](https://github.com/PAIR-code/lit/blob/main/lit_nlp/api/layout.py#L173) を参照してください。

### LIME による説明の探索
それでは、先ほどレンダリングした UI を使用して LIME による説明を調べてみましょう。

[LIME](https://arxiv.org/abs/1602.04938) は、広く使用されているブラックボックス（またはモデル非依存）の説明可能性手法です。ブラックボックス手法であるため、あらゆる ML モデル（より一般的には入力と出力を持つ任意の関数）に対して LIME を使用して重要度（saliency）を取得できます。

1. 左パネルからテキスト入力を選択します。
2. 右上パネルで、`negative` クラスと `positive` クラスの予測確率、正解ラベル（`T` で表示）、および予測クラス（`P`）を確認します。
3. 右下パネルで、`Explanations` タブを選択します。
4. `LIME` の右側にある矢印記号をクリックします。
5. 各単語に重要度スコアが表示されます。色が薄すぎる場合は、色がはっきりと見えるまで `Gamma` スコアを上げてください。
6. 予測されたクラスがすでに `Class to explain`（説明対象のクラス）として設定されていることを確認します。
7. （鮮やかな赤や青で着色された）重要な単語にマウスカーソルを合わせ、スコアを確認します。正のスコアはそのラベル（下のスクリーンショットでは `positive` クラス）への正の寄与を意味し、負のスコアは負の寄与を意味します。

<img width="800" src="https://github.com/user-attachments/assets/05623753-bdf8-4aec-88c6-3e7d0718576a">


---

## 勾配ベースの重要度（Saliency）手法の追加

LIME のようなブラックボックス手法は、モデルの内部へのアクセスを必要としないため柔軟で使いやすいです。しかし、勾配などのモデル内部の情報を活用することで、モデルの挙動をより繊細に理解できます。

勾配ベースの解釈可能性の核心となるアイデアは、各入力 $x$ に対する出力 $y$ の勾配を計算することです。

テキスト分類では、出力はモデルの最終層によって生成されるロジットであり、入力はトークン埋め込み（token embeddings）です（画像分類の場合、入力は画像のピクセルになります）。

### モデルの分析

BERT モデルを確認し、この目的のために分析してみましょう。

In [ ]:
keras.utils.plot_model(classifier, show_shapes=True, show_layer_names=True)

モデル内のレイヤを確認してみましょう。

In [ ]:
classifier.layers

メインのモデルは `BertBackbone` レイヤ（`classifier.layers[3]`）内にネストされているようです。

In [ ]:
classifier.layers[3].layers

アーキテクチャを確認するために、ネストされた `BertBackbone` をプロットしてみましょう。

In [ ]:
keras.utils.plot_model(
    classifier.layers[3], show_shapes=True, show_layer_names=True
)

トークン埋め込みを取得したいと考えます。`token_embedding` レイヤ（`classifier.layers[3].layers[1]`）の出力がトークン埋め込みを生成しているようです。ただし、最終的な埋め込みは、位置エンコーディング（positional encoding）を結合した `add_1` レイヤ（`classifier.layers[3].layers[5]`）によって生成されます。<br>
また、その後に Layer Normalization レイヤ（`classifier.layers[3].layers[6]`）が続きます。この埋め込みは、勾配ベースの解釈可能性にとって適切な対象であると考えられます。

このレイヤまでのスタンドアロン モジュールを定義しましょう。ここではネストされたレイヤを見つけることができます。グラフを可視化してみましょう。

In [ ]:
embedding_model = keras.Model(
    inputs=classifier.input, outputs=classifier.layers[3].layers[6].output
)

In [ ]:
keras.utils.plot_model(embedding_model)

その後にメインの BERT モデルが続きます。ここでは同様に、残りの BERT バックボーンをスタンドアロン モジュールとして定義します。

In [ ]:
bert_backbone = keras.Model(
    inputs=[
        classifier.layers[3].layers[7].input,
        classifier.layers[3].layers[8].output,
    ],
    outputs=classifier.layers[3].layers[-1].output,
)

In [ ]:
keras.utils.plot_model(bert_backbone)

最後に、勾配の計算元となる最終的なロジット値を生成する分類ヘッドを定義します。

In [ ]:
cls_head = keras.Model(
    inputs=classifier.layers[-2].input, outputs=classifier.layers[-1].output
)

In [ ]:
keras.utils.plot_model(cls_head)

これら 3 つのモジュールを順番に呼び出して、テキストからロジットを計算できます。

In [ ]:
data = ImdbSentimentData(test_ds)
text = data.examples[0]["text"]

tokens = classifier._preprocessor(keras.ops.convert_to_tensor([text]))
embeddings = embedding_model(tokens)
bert_output = bert_backbone([embeddings, tokens["padding_mask"]])
logits = cls_head(bert_output)
logits

### 勾配の計算
TensorFlow では、`tf.GradientTape` コンテキスト マネージャーを使用して自動微分（autograd）機能を利用できます。

他のフレームワークを使用している場合は、同じ手順を行うために autograd に関するドキュメントを参照してください（例: [PyTorch](https://pytorch.org/tutorials/beginner/former_torchies/autograd_tutorial.html), [JAX](https://jax.readthedocs.io/en/latest/notebooks/autodiff_cookbook.html)）。

In [ ]:
with tf.GradientTape() as tape:
    tape.watch(embeddings)
    bert_output = bert_backbone([embeddings, tokens["padding_mask"]])
    logits = cls_head(bert_output)

    indices = tf.argmax(logits, axis=-1)
    pred = tf.gather_nd(logits, [[i, idx] for i, idx in enumerate(indices)])

input_emb_grads = tape.gradient(pred, embeddings)
input_emb_grads

### LIT モデルの更新

素晴らしいですね。

このセットアップに基づいて、勾配ノルム（Gradient Norm）と勾配と入力の内積（Gradient-dot-Input）という 2 つの勾配ベースの手法を追加できます。

#### 勾配ノルム（Gradient Norm）
勾配ノルムは、重要度スコアが勾配の L2 ノルムに比例するシンプルな手法です。すなわち、トークン $i$ のスコアは次のようになります。

$S(i)\propto  \|  
abla _{x_i} \hat{y} \| _2$

#### 勾配と入力の内積（Gradient-dot-Input）
Gradient-dot-Input では、重要度スコアは入力埋め込みとその勾配の内積に比例します。すなわち、トークン $i$ について以下を計算します。

$S(i)\propto  x_i 
abla_{x_i} \hat{y} $


前のモデルですでに $\hat{y}$ を定義しているため、生成する必要がある追加の要素は $
abla _{x_i}$ と $x_i$ の 2 つです。
- $x_i$: トークン埋め込み。これは Gradient-dot-Input にのみ必要です。
- $
abla _{x_i}$: 各トークン埋め込みの勾配。これは Gradient Norm と Gradient-dot-Input の両方に必要です。

これらの要素を反映するように LIT モデル モジュールを更新しましょう。

主な更新点は以下のとおりです。
- `__init__()` 内で、上記で説明した追加モジュールを定義します。
- `output_spec()` の出力に、次の 3 つの追加のキーと値の仕様が含まれるようになります。
  - `"tokens"`: 形状 `(num_tokens)` のトークンのリスト。仕様には、入力テキストとリンクするために `parent="text"` を指定した `Tokens` LIT 型を使用します。
  - `"input_embedding"`: 形状 `(num_tokens, emb_dim)` のトークン埋め込み（$x_i$）。仕様には、`align="tokens"` を指定した `TokenEmbeddings` LIT 型を使用します。
  - `"input_emb_grad"`: 形状 `(num_tokens, emb_dim)` のトークン埋め込みの勾配（$
abla _{x_i}$）。仕様には、リンクを定義するために `align="tokens"` および `grad_for="input_embedding"` を指定した `TokenGradients` LIT 型を使用します。
- `predict()` は、上記で説明した 3 つのサブモジュールと `tf.GradientTape` を使用して勾配を計算し、`output_spec()` の定義に対応する 3 つの追加要素を返すようになります。<br>
なお、`tokens`、`input_embedding`、および `input_emb_grad` を `[START_OFFSET : slen - END_OFFSET]` でスライスしていることに注意してください。ここで:
  - `START_OFFSET=1`: 入力文のどの要素とも対応しない先頭の `[CLS]` トークンをトリミングします。
  - `END_OFFSET=1`: 最初の文の後に BERT が使用する `[SEP]` トークンをトリミングします。BERT は文のペアを受け取ることができ、`[SEP]` は文の間の区切り文字ですが、このテキスト分類の例では単一の文のみを使用しているためです。
  - `slen`: 入力トークンの長さです。

In [ ]:
START_OFFSET = 1  # [CLS] トークンのオフセット
END_OFFSET = 1  # [SEP] トークンのオフセット


class BertSentimentModel(lit_model.Model):

    def __init__(self, model):
        self._model = model
        ########## モデルのスライス ##########
        self._embedding_model = keras.Model(
            inputs=model.input, outputs=model.layers[3].layers[6].output
        )
        self._bert_backbone = keras.Model(
            inputs=[
                model.layers[3].layers[7].input,
                model.layers[3].layers[8].output,
            ],
            outputs=model.layers[3].layers[-1].output,
        )
        self._cls_head = keras.Model(
            inputs=classifier.layers[-2].input,
            outputs=classifier.layers[-1].output,
        )
        ######################################

    def _detokenize(self, indices):
        return [
            self._model.preprocessor.tokenizer.vocabulary[idx]
            for idx in indices
        ]

    def predict(self, inputs):
        """Predict on a stream of examples."""
        examples = [d["text"] for d in inputs]
        tokenized = self._model.preprocessor(examples)
        seq_lens = keras.ops.sum(
            keras.ops.cast(tokenized["padding_mask"], "int32"), axis=-1
        )
        embeddings = self._embedding_model(tokenized)

        ########## 勾配の計算 ##########
        with tf.GradientTape() as tape:
            tape.watch(embeddings)
            bert_output = self._bert_backbone(
                [embeddings, tokenized["padding_mask"]]
            )
            logits = self._cls_head(bert_output)

            if "label" in inputs[0].keys():
                indices = [
                    1 if inp["label"] == LABELS[1] else 0 for inp in inputs
                ]
            else:
                indices = tf.argmax(logits, axis=-1)

            pred = tf.gather_nd(
                logits, [[i, idx] for i, idx in enumerate(indices)]
            )

        probs = keras.activations.softmax(logits)

        input_emb_grads = tape.gradient(pred, embeddings)
        ################################

        return [
            {
                "prob": prob.numpy(),
                ########## 新しい出力 ##########
                "tokens": self._detokenize(
                    tokens[START_OFFSET : slen - END_OFFSET]
                ),
                "input_embedding": emb[
                    START_OFFSET : slen - END_OFFSET
                ].numpy(),
                "input_emb_grad": grad[
                    START_OFFSET : slen - END_OFFSET
                ].numpy(),
                #################################
            }
            for prob, tokens, emb, grad, slen in zip(
                probs,
                tokenized["token_ids"],
                embeddings,
                input_emb_grads,
                seq_lens,
            )
        ]

    def input_spec(self):
        """Describe the inputs to the model."""
        return {
            "text": lit_types.TextSegment(),
            "label": lit_types.CategoryLabel(required=False, vocab=LABELS),
        }

    def output_spec(self):
        """Describe the model outputs."""
        return {
            "prob": lit_types.MulticlassPreds(vocab=LABELS, parent="label"),
            ########## 新しい出力仕様 ##########
            "tokens": lit_types.Tokens(parent="text"),
            "input_embedding": lit_types.TokenEmbeddings(align="tokens"),
            "input_emb_grad": lit_types.TokenGradients(
                align="tokens", grad_for="input_embedding"
            ),
            ####################################
        }

もう一度 LIT をレンダリングして、これらの変更が UI にどのように反映されるか確認してみましょう。

In [ ]:
# 分析対象のモデルとデータセットを指定して LIT ウィジェットを作成します。
models = {"BERT classifier": BertSentimentModel(classifier)}

widget.stop()
widget = notebook.LitWidget(models, datasets, layouts=layouts, port=8890)

In [ ]:
widget.render(height=600, open_in_new_tab=True)

## 統合勾配（Integrated Gradients）の追加

統合勾配法（Integrated Gradients）は、埋め込み空間内のパスに沿って勾配を積分することに基づき、特徴量の寄与を推定するためのより堅牢な手法です。<br>
アルゴリズムの詳細については、[Sundararajan et al. 2017](https://arxiv.org/abs/1703.01365) を参照してください。この手法は grad-norm や grad-dot-input よりも優れた結果をもたらす可能性がありますが、モデルへのより詳細な計測（instrumentation）も必要とします。

統合勾配を使用するための主な更新点は以下のとおりです。
- `input_spec()` に、出力の `input_embedding` 仕様と同じ名前の、オプションのキーと値（`required=False`）として `input_embedding` が含まれるようになりました。<br>
統合勾配では、LIT は勾配を計算するための複数の入力（この場合はトークン埋め込み）摂動を計算し、それらを積分します。このオプションの入力はそれらの摂動のゲートであり、統合勾配を使用する場合にのみ渡されます。
- `output_spec()` は、入力からの label フィールドとリンクするための `grad_target_field_key="label"` が追加されている点を除き、前の例とほぼ同じです。
- `predict()` に統合勾配用の条件付き処理が追加されました（つまり、入力に `input_embedding` キーが含まれている場合）。元の入力から計算された生の埋め込みを摂動で置き換えるだけです。

In [ ]:
START_OFFSET = 1  # [CLS] トークンのオフセット
END_OFFSET = 1  # [SEP] トークンのオフセット


class BertSentimentModel(lit_model.Model):

    def __init__(self, model):
        self._model = model
        self._embedding_model = keras.Model(
            inputs=model.input, outputs=model.layers[3].layers[6].output
        )
        self._bert_backbone = keras.Model(
            inputs=[
                model.layers[3].layers[7].input,
                model.layers[3].layers[8].output,
            ],
            outputs=model.layers[3].layers[-1].output,
        )
        self._cls_head = keras.Model(
            inputs=classifier.layers[-2].input,
            outputs=classifier.layers[-1].output,
        )

    def _detokenize(self, indices):
        return [
            self._model.preprocessor.tokenizer.vocabulary[idx]
            for idx in indices
        ]

    def predict(self, inputs):
        """Predict on a stream of examples."""
        examples = [d["text"] for d in inputs]
        tokenized = self._model.preprocessor(examples)
        seq_lens = keras.ops.sum(
            keras.ops.cast(tokenized["padding_mask"], "int32"), axis=-1
        )
        embeddings = self._embedding_model(tokenized)
        ########## IG 摂動用の条件付き処理 ##########
        if "input_embedding" in inputs[0].keys():
            pertubations = keras.ops.convert_to_tensor(
                [inp["input_embedding"] for inp in inputs]
            )
            step_size, ptb_len, emd_dim = pertubations.shape
            pertubations = keras.ops.reshape(
                pertubations, (step_size * ptb_len, emd_dim)
            )
            update_indices = [
                [s, l + START_OFFSET]
                for s in range(step_size)
                for l in range(ptb_len)
            ]
            embeddings = tf.tensor_scatter_nd_update(
                embeddings, update_indices, pertubations
            )
        #############################################

        with tf.GradientTape() as tape:
            tape.watch(embeddings)
            bert_output = self._bert_backbone(
                [embeddings, tokenized["padding_mask"]]
            )
            logits = self._cls_head(bert_output)
            if "label" in inputs[0].keys():
                indices = [
                    1 if inp["label"] == LABELS[1] else 0 for inp in inputs
                ]
            else:
                indices = tf.argmax(logits, axis=-1)
            pred = tf.gather_nd(
                logits, [[i, idx] for i, idx in enumerate(indices)]
            )

        probs = tf.keras.activations.softmax(logits)

        input_emb_grads = tape.gradient(pred, embeddings)

        return [
            {
                "prob": prob.numpy(),
                "tokens": self._detokenize(
                    tokens[START_OFFSET : slen - END_OFFSET]
                ),
                "input_embedding": emb[
                    START_OFFSET : slen - END_OFFSET
                ].numpy(),
                "input_emb_grad": grad[
                    START_OFFSET : slen - END_OFFSET
                ].numpy(),
            }
            for prob, tokens, emb, grad, slen in zip(
                probs,
                tokenized["token_ids"],
                embeddings,
                input_emb_grads,
                seq_lens,
            )
        ]

    def input_spec(self):
        """Describe the inputs to the model."""
        return {
            "text": lit_types.TextSegment(),
            "label": lit_types.CategoryLabel(required=False, vocab=LABELS),
            ########## IG 摂動用の新しい入力仕様 ##########
            "input_embedding": lit_types.TokenEmbeddings(
                required=False, align="tokens"
            ),
            ###############################################
        }

    def output_spec(self):
        """Describe the model outputs."""
        return {
            "prob": lit_types.MulticlassPreds(vocab=LABELS, parent="label"),
            "tokens": lit_types.Tokens(parent="text"),
            "input_embedding": lit_types.TokenEmbeddings(align="tokens"),
            "input_emb_grad": lit_types.TokenGradients(
                align="tokens",
                grad_for="input_embedding",
                ########## IG 用の追加引数 ##########
                grad_target_field_key="label",
                #####################################
            ),
        }

In [ ]:
# 分析対象のモデルとデータセットを指定して LIT ウィジェットを作成します。
models = {"BERT classifier": BertSentimentModel(classifier)}

widget.stop()
widget = notebook.LitWidget(models, datasets, layouts=layouts, port=8890)

もう一度 LIT UI をレンダリングしてみましょう。

統合勾配（Integrated Gradients）は追加の計算を必要とするため、矢印をクリックして統合勾配による重要度を計算します。

また、設定ボタンからステップサイズ（積分を近似する摂動の数）を 5 から 100 の間で変更できます。値が大きいほど精度が高くなりますが、計算コストも高くなります。
<img width="700" alt="image" src="https://github.com/user-attachments/assets/1f41aaeb-df8a-4d06-9191-de7853f1bfc0">

In [ ]:
widget.render(height=600, open_in_new_tab=True)

## （オプション）解釈可能性モジュールの直接呼び出し

LIT UI の外部で重要度（saliency）の値を直接取得したい場合があります。同じモデル オブジェクトを使用して各コンポーネントを呼び出すことができます。

たとえば、統合勾配（Integrated Gradients）の場合は次のようになります。

In [ ]:
from lit_nlp.components import gradient_maps

ig = gradient_maps.IntegratedGradients()

saliency = ig.run(
    ImdbSentimentData(test_ds).examples[:1],
    BertSentimentModel(classifier),
    ImdbSentimentData(test_ds),
)
saliency

In [ ]:
df = pd.DataFrame(
    [
        saliency[0]["input_emb_grad"].tokens,
        saliency[0]["input_emb_grad"].salience,
    ],
    index=["token", "salience"],
)

salience_series = saliency[0]["input_emb_grad"].salience
abs_max = max(abs(salience_series.max()), abs(salience_series.min()))
df.style.background_gradient(
    axis=1,
    gmap=df.loc["salience", :].apply(pd.to_numeric),
    vmin=-abs_max,
    vmax=abs_max,
    cmap="seismic",
)

## 次のステップ

LIT および AI Explanations の詳細については、以下のリソースを確認してください。

- [LIT ドキュメント](https://pair-code.github.io/lit/)
- [LIT GitHub](https://github.com/PAIR-code/lit)
- [統合勾配（Integrated Gradients）の論文](https://arxiv.org/abs/1703.01365)

Copyright 2025 Google Inc. Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0 Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License